In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import hashlib
import gzip
import numpy as np
import re
import os
from pandas.api.types import CategoricalDtype

In [ ]:
def map_assembler_config(cell):
    if "carpedeam" in cell and "configUnsafe" in cell:
        return "CarpeDeam\n(unsafe mode)"
    elif "carpedeam" in cell and "configSafe" in cell:
        return "CarpeDeam\n(safe mode)"
    elif "penguin" in cell:
        return "PenguiN"
    elif "megahit" in cell:
        return "MEGAHIT"
    elif "spades" in cell:
        return "metaSPAdes"
    else:
        return cell  # Return the cell as is if none of the conditions are met

In [ ]:
def extractInfo(file_name):
    # Regular expression to extract components
    pattern = r"(?P<sample>\w+)_d(?P<damage>\d+)_f(?P<fragmentLen>\d+)_cov(?P<coverage>\d+)\.raw-raw\.assm\.(?P<config>.+)\.report\.tsv"
    
    # Match the pattern
    match = re.match(pattern, file_name)
    
    # Extract components if the pattern matches
    if match:
        sample = match.group("sample")
        damage = match.group("damage")
        fragmentLen = match.group("fragmentLen")
        coverage = match.group("coverage")
        config = match.group("config")
        return sample, damage, fragmentLen, coverage, config
    return None, None, None, None, None

In [ ]:
def map_damage(cell):
    if "1" in cell:
        return "Moderate___"  # Fixed length with padding
    elif "2" in cell:
        return "High_______"
    elif "3" in cell:
        return "Ultra-High_"
    else:
        return cell  # Return the cell as is if none of the conditions are met

In [ ]:
def map_fraglen(cell):
    if "1" in cell:
        return "Short_______"
    elif "2" in cell:
        return "Ultra-Short_"
    elif "3" in cell:
        return "Medium______"
    else:
        return cell  # Return the cell as is if none of the conditions are met

In [ ]:
def map_cov(cell):
    if "3" in cell:
        return "_3X"
    elif "5" in cell:
        return "_5X"
    elif "10" in cell:
        return "10X"
    else:
        return cell  # Return the cell as is if none of the conditions are met

In [ ]:
def standardize_columns(df):
    """
    Standardizes column names in a DataFrame.
    """
    df.columns = (
        df.columns.str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace(">=", "ge", regex=False)
        .str.replace("<=", "le", regex=False)
        .str.replace("<", "lt", regex=False)
        .str.replace(">", "gt", regex=False)
        .str.replace("=", "eq", regex=False)
        .str.replace("%", "perc", regex=False)
        .str.replace("#", "num", regex=False)
    )
    return df

In [ ]:
def curate_df(path_tsv, cois):
    files = glob.glob(path_tsv)
    dfs = []
    
    for file in files:
        
        if any(conf in file for conf in cois):
        
            df = pd.read_csv(file, sep='\t')

            # Standardize column names
            df = standardize_columns(df)

            # Extract information from the filename
            name = os.path.basename(file)
            sample, damage, fragmentLen, coverage, config = extractInfo(name)

            # Add extracted information as columns
            df["sample"] = sample
            df["damage"] = damage
            df["fragmentlen"] = fragmentLen
            df["coverage"] = coverage
            df["config"] = config
            df["assembler"] = df["config"].apply(map_assembler_config)

            dfs.append(df)
    
    big_df = pd.concat(dfs, ignore_index=True)
    
    # Add additional computed columns
    big_df["mis_per_contig"] = big_df["num_misassemblies"] / big_df["num_contigs_ge_0_bp"]
    big_df["mis_per_aln_base"] = big_df["num_misassemblies"] / big_df["total_aligned_length"]
    
    return big_df


In [ ]:
def plot_heatmap(big_df, metric, numType):
    """
    Create heatmaps for a specific metric, with distinct color gradients per sample.
    Assemblers are on the x-axis, and parameter combinations are on the y-axis.
    
    Args:
        big_df: The large DataFrame with metrics.
        metric: Metric to display (no aggregation).
    """
    
    # Define the custom order for the parameter combinations
    custom_order = [
        "Moderate_Medium_3X", "Moderate_Medium_5X", "Moderate_Medium_10X",
        "Moderate_Short_3X", "Moderate_Short_5X", "Moderate_Short_10X",
        "Moderate_Ultra-Short_3X", "Moderate_Ultra-Short_5X", "Moderate_Ultra-Short_10X",
        "High_Medium_3X", "High_Medium_5X", "High_Medium_10X",
        "High_Short_3X", "High_Short_5X", "High_Short_10X",
        "High_Ultra-Short_3X", "High_Ultra-Short_5X", "High_Ultra-Short_10X",
        "Ultra-High_Medium_3X", "Ultra-High_Medium_5X", "Ultra-High_Medium_10X",
        "Ultra-High_Short_3X", "Ultra-High_Short_5X", "Ultra-High_Short_10X",
        "Ultra-High_Ultra-Short_3X", "Ultra-High_Ultra-Short_5X", "Ultra-High_Ultra-Short_10X"
    ]

    # Find the maximum lengths of the first, second, and last parts
    max_first = max(len(item.split('_')[0]) for item in custom_order)
    max_second = max(len(item.split('_')[1]) for item in custom_order)
    max_last = max(len(item.split('_')[2]) for item in custom_order)

    # Pad each entry based on the maximum lengths of the first, second, and last parts
    padded_custom_order = [
        f"{item.split('_')[0].ljust(max_first, '_')}_"
        f"{item.split('_')[1].ljust(max_second, '_')}_"
        f"{item.split('_')[2].rjust(max_last, '_')}"
        for item in custom_order
    ]


    metric_dict = {
        "largest_alignment": "Largest Alignment",
        "genome_fraction_perc": "Genome Fraction (%)",
        "mis_per_contig": "# Misassemblies per Contig",
        "mis_per_aln_base": "Misassemblies per aligned bp",
        "na50": "NA50"
    }

    sample_dict = {
        "gut": "Gut",
        "calc": "Calculus",
        "bone": "Bone"
    }
    
    big_df["parameter_combination"] = (
        big_df["damage"].apply(map_damage) +
        big_df["fragmentlen"].apply(map_fraglen) +
        big_df["coverage"].apply(map_cov)
    )

    # Generate a unique colormap for each sample
    unique_samples = big_df["sample"].unique()
    colormaps = sns.color_palette("coolwarm", as_cmap=True)
    
    for i, sample in enumerate(unique_samples):
        sample_df = big_df[big_df["sample"] == sample]
        
        # Pivot table with parameter combinations as rows and assemblers as columns
        pivot_table = sample_df.pivot(
            index="parameter_combination",
            columns="assembler",
            values=metric
        )
        
        # Sort the pivot table based on the custom order
        pivot_table = pivot_table.reindex(padded_custom_order, axis=0)

        # Round the metric values to integers for better readability
        if numType == "d":
            pivot_table = pivot_table.round(0)
        else:
            pivot_table = pivot_table.round(3)
    
        # Create the heatmap
        plt.figure(figsize=(10, 8))  # Adjust size as needed
        ax = sns.heatmap(
            pivot_table,
            annot=True,
            fmt=numType,  # Integer format
            cmap=colormaps,  # Unique colormap per sample
            cbar_kws={"label": metric_dict[metric]},
            linewidths=0.5
        )
        
        # Set the y-axis tick labels to use monospace font
        ax.set_yticklabels(ax.get_yticklabels(), fontname="monospace")
        
        # Adjust the font size of the color bar legend
        cbar = ax.collections[0].colorbar
        cbar.ax.yaxis.label.set_size(14)
        
        # Add a secondary legend to explain y-label structure
        plt.figtext(
            0.02, 0.98,  # Position: (x, y) in figure coordinates
            "Damage   FragLen   Cov",
            fontsize=12,
            fontname="monospace",
            va="top",
            ha="left"
        )
        
        metric_clean = metric_dict[metric]
        sample_clean = sample_dict[sample]
        
        plt.title(f"{metric_clean} for {sample_clean} Sample")
        plt.xlabel("Assembler", fontsize=14)
        plt.ylabel("", fontsize=14)
        plt.tight_layout()
        plt.show()

In [ ]:
configsOfInterest = ['megahit.config10', 'carpedeam.configSafe3', 'carpedeam.configUnsafe3', 'penguin.config10', 'spades.config13'] 

In [ ]:
path="data/sim/results/assembly-evaluation-quast/*tsv"

In [ ]:
revi = curate_df(path, configsOfInterest)

In [ ]:
for m in [("na50", "d"), ("mis_per_contig", ".3f"), ("largest_alignment", "d"), ("genome_fraction_perc", ".1f")]:
    plot_heatmap(revi, m[0], m[1])